# 07 – Evaluasi akhir (CP7, FR-10, FR-13)

Perbandingan utama (ditetapkan sebelum eksperimen): **S6 lawan S1 dan S2** pada AUROC dan F1, dilaporkan apa pun arahnya. Semua perbandingan lain berlabel *eksploratif*. Setelah CP7 angka tidak diubah; koreksi berarti mengulang dari checkpoint yang terdampak.

**Notebook mandiri (Versi 2):** tanpa folder `src`, tanpa Docker. Semua kode ada di sel-sel notebook ini. Kode ditulis dari Versi 1 dan **belum dijalankan** di mesin penulis (tanpa torch/GPU); jalankan dari atas ke bawah di mesin RTX 3070 Ti. Hasil yang tidak dapat dihitung dilaporkan apa adanya, bukan diisi.

## 1. Pustaka tambahan (opsional)

In [ ]:
INSTALL = False      # True hanya sekali, di venv yang sama dengan notebook 00 (torch CUDA sudah terpasang di sana)

if INSTALL:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn", "scipy", "pandas", "pyyaml", "tabulate"])

## 2. Impor, konfigurasi, utilitas, dan evaluasi

In [ ]:
from __future__ import annotations

import contextlib, copy, datetime as dt, gc, hashlib, importlib, io, json, math, os, platform, random, re, shutil, subprocess, sys, tempfile, time
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import yaml
from IPython.display import display, Image

# expandable_segments mengurangi fragmentasi memori, tetapi belum didukung di Windows.
if platform.system() != "Windows":
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
import torch.nn as nn
import torch.nn.functional as F


def find_root() -> Path:
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "configs" / "base.yaml").exists():
            return q
    raise FileNotFoundError("configs/base.yaml tidak ditemukan dari direktori kerja ke atas")


ROOT = find_root()
os.chdir(ROOT)

_ENV = {"LARYNX_DATASET": "dataset", "LARYNX_OUTPUTS": "outputs", "LARYNX_HF_CACHE": "hf_cache"}


def deep_merge(a: dict, b: dict) -> dict:
    out = copy.deepcopy(a)
    for k, v in b.items():
        out[k] = deep_merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else copy.deepcopy(v)
    return out


def _load(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    base = cfg.pop("_base", None)
    return deep_merge(_load(path.parent / base), cfg) if base else cfg


def load_merged(path) -> dict:
    cfg = _load(Path(path))
    for env, key in _ENV.items():
        if os.environ.get(env):
            cfg["paths"][key] = os.environ[env]
    for key, val in cfg["paths"].items():           # jalur relatif -> mutlak terhadap folder Versi 2
        if not Path(val).is_absolute():
            cfg["paths"][key] = str((ROOT / val).resolve())
    return cfg


def P(cfg: dict, *parts: str) -> Path:
    return Path(cfg["paths"]["outputs"]).joinpath(*parts)


def config_hash(cfg: dict) -> str:
    return hashlib.sha256(json.dumps(cfg, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")).hexdigest()


def job_seed(base_seed: int, *parts) -> int:
    key = "|".join([str(base_seed), *map(str, parts)]).encode("utf-8")
    return int.from_bytes(hashlib.sha256(key).digest()[:4], "big")


def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)


def atomic_write_bytes(path, data: bytes) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=path.parent, prefix=path.name + ".", suffix=".tmp")
    try:
        with os.fdopen(fd, "wb") as f:
            f.write(data); f.flush(); os.fsync(f.fileno())
        os.replace(tmp, path)
    except BaseException:
        if os.path.exists(tmp):
            os.remove(tmp)
        raise


def atomic_write_json(path, obj) -> None:
    atomic_write_bytes(path, json.dumps(obj, indent=2, ensure_ascii=False, sort_keys=True, default=str).encode("utf-8"))


def atomic_torch_save(obj, path) -> None:
    buf = io.BytesIO(); torch.save(obj, buf); atomic_write_bytes(path, buf.getvalue())


def is_done(job_dir, cfg_hash: str) -> bool:
    done = Path(job_dir) / "DONE"
    return done.exists() and done.read_text(encoding="utf-8").strip() == cfg_hash


def mark_done(job_dir, cfg_hash: str) -> None:
    atomic_write_bytes(Path(job_dir) / "DONE", cfg_hash.encode("utf-8"))


def git_commit() -> str:
    # Versi 2 tanpa Docker: hash commit dari git (bila ada), ditambah -dirty bila ada perubahan belum di-commit.
    try:
        c = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], capture_output=True, text=True, timeout=15).stdout.strip()
        d = subprocess.run(["git", "-C", str(ROOT), "status", "--porcelain"], capture_output=True, text=True, timeout=15).stdout.strip()
        return (c or "tanpa-commit") + ("-dirty" if d and c else "")
    except Exception:  # noqa: BLE001
        return "tanpa-commit"


GIT_COMMIT = git_commit()
CFG_PATH = "configs/base.yaml"
cfg = load_merged(CFG_PATH)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = P(cfg, "cp0")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT   :", ROOT)
print("dataset:", cfg["paths"]["dataset"])
print("outputs:", cfg["paths"]["outputs"])
print("device :", DEV, "| commit:", GIT_COMMIT, "| config_hash:", config_hash(cfg)[:12])

def free_gpu() -> None:
    # Pengganti 'proses baru per job' di Versi 1: dipanggil di akhir tiap job agar memori GPU kembali bersih.
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


WORK_DIR = Path(cfg["paths"]["outputs"]) / "_workers"


def install_module(name: str, src: str):
    # Windows memakai 'spawn': fungsi/kelas yang dipakai proses anak harus berada di modul yang dapat diimpor. Modul dibuat
    # saat notebook berjalan (satu-satunya sumber kodenya ada di sel notebook ini) lalu diimpor, termasuk oleh proses anak.
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    p = WORK_DIR / f"{name}.py"
    changed = (not p.exists()) or p.read_text(encoding="utf-8") != src
    if changed:
        p.write_text(src, encoding="utf-8")
    if str(WORK_DIR) not in sys.path:
        sys.path.insert(0, str(WORK_DIR))
    os.environ["PYTHONPATH"] = str(WORK_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")
    importlib.invalidate_caches()
    mod = importlib.import_module(name)
    return importlib.reload(mod) if changed and name in sys.modules else mod

In [ ]:
# ---------------------------------------------------------------- checkpoint, persetujuan manusia, LOG.md
# Pengganti checkpoint.py dan approve.py Versi 1. Persetujuan MANUSIA tidak pernah dipanggil otomatis oleh notebook.
LULUS, GAGAL, MENUNGGU, TIDAK_TERSEDIA = "LULUS", "GAGAL", "MENUNGGU", "TIDAK_TERSEDIA"
DEC_PATH, BASE_PATH, LOG_PATH = ROOT / "configs" / "decisions.yaml", ROOT / "configs" / "base.yaml", ROOT / "LOG.md"


def file_sha256(path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def write_manifest(out_dir, files, extra=None) -> Path:
    out_dir = Path(out_dir)
    entries = {}
    for f in files:
        p = Path(f)
        if not p.exists():
            raise FileNotFoundError(f"Keluaran checkpoint tidak ditemukan: {p}")
        key = str(p.relative_to(out_dir)) if p.is_relative_to(out_dir) else str(p)
        entries[key] = {"sha256": file_sha256(p), "bytes": p.stat().st_size}
    mp = out_dir / "manifest.json"
    atomic_write_json(mp, {"files": entries, **(extra or {})})
    return mp


def _item(nama: str, status: str, detail: str) -> dict:
    return {"item": nama, "status": status, "detail": detail}


def _review_item(out_dir) -> dict:
    out_dir = Path(out_dir)
    ok = (out_dir / "REVIEWED").exists()
    return _item("Tinjauan orang kedua (berkas REVIEWED)", LULUS if ok else MENUNGGU,
                 "ada" if ok else f"setelah meninjau hasil: approve_review('{out_dir.name.upper()}', 'NAMA') (RULES §3)")


def _flag_item(out_dir, name: str, nama: str, how: str) -> dict:
    ok = (Path(out_dir) / name).exists()
    return _item(nama, LULUS if ok else MENUNGGU, "ada" if ok else how)


def _decision_item(d: str) -> dict:
    with open(DEC_PATH, "r", encoding="utf-8") as f:
        dec = yaml.safe_load(f)[d]
    st = dec["status"]
    status = {"diputuskan": LULUS, "ditunda": LULUS, "menunggu_konfirmasi": MENUNGGU}.get(st, GAGAL)
    return _item(f"{d} {st}", status, str(dec.get("isi", "")))


def _finish(cp: str, out_dir, items: list, files: list, extra=None) -> dict:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    blocking = [i for i in items if i["status"] in (GAGAL, MENUNGGU)]
    rep = {"checkpoint": cp, "lulus": not blocking, "items": items, "penghambat": [i["item"] for i in blocking]}
    atomic_write_json(out_dir / f"{cp.lower()}_check.json", rep)
    for i in items:
        print(f"[{i['status']:>15}] {i['item']} -- {i['detail']}")
    if rep["lulus"]:
        write_manifest(out_dir, [out_dir / f"{cp.lower()}_check.json", *[f for f in files if Path(f).exists()]],
                       extra={"checkpoint": cp, "git_commit": GIT_COMMIT, **(extra or {})})
        print(f"\n{cp} LULUS. manifest.json ditulis. Tag git '{cp.lower()}' dapat dibuat (setelah tinjauan orang kedua).")
    else:
        print(f"\n{cp} BELUM LULUS. Penghambat: {rep['penghambat']}\nmanifest.json TIDAK ditulis.")
    return rep


def log_row(who: str, what: str, hasil: str = "disetujui", keputusan: str = "via notebook") -> None:
    line = f"| {dt.date.today().isoformat()} | {who} | {what} | {hasil} | {keputusan} |\n"
    if LOG_PATH.exists():
        with open(LOG_PATH, "a", encoding="utf-8") as f:
            f.write(line)


def approve_flag(cp: str, name: str, by: str, what: str) -> None:
    d = P(load_merged(BASE_PATH), cp.lower())
    d.mkdir(parents=True, exist_ok=True)
    (d / name).write_text(f"{by} {dt.datetime.now().isoformat(timespec='seconds')}\n", encoding="utf-8")
    log_row(by, f"{what} ({cp})")
    print("ditulis", d / name)


def approve_review(cp: str, by: str) -> None:
    approve_flag(cp, "REVIEWED", by, "Tinjauan orang kedua")


def approve_decisions(ids: list, by: str) -> None:
    text = DEC_PATH.read_text(encoding="utf-8")
    for d in ids:
        pat = re.compile(rf"(^{d}:\n  status: )menunggu_konfirmasi", re.M)
        if not pat.search(text):
            print(f"{d}: tidak berstatus menunggu_konfirmasi (dilewati)")
            continue
        text = pat.sub(rf"\1diputuskan\n  dikonfirmasi_oleh: {by}\n  tanggal_konfirmasi: {dt.date.today().isoformat()}", text)
        log_row(by, f"Konfirmasi keputusan {d}")
        print(f"{d}: diputuskan oleh {by}")
    DEC_PATH.write_text(text, encoding="utf-8")


def approve_roi(by: str) -> None:
    t = BASE_PATH.read_text(encoding="utf-8")
    t2 = re.sub(r"(  roi:.*?\n    locked: )false", r"\1true", t, count=1, flags=re.S)
    if t2 == t and "locked: true" not in t:
        raise SystemExit("preprocess.roi.locked tidak ditemukan di base.yaml")
    BASE_PATH.write_text(t2, encoding="utf-8")
    d = P(load_merged(BASE_PATH), "cp2")
    d.mkdir(parents=True, exist_ok=True)
    (d / "ROI_APPROVED").write_text(f"{by} {dt.datetime.now().isoformat(timespec='seconds')}\n", encoding="utf-8")
    log_row(by, "Parameter Frangi dikunci setelah pemeriksaan visual 50 citra (FR-03)")
    print("preprocess.roi.locked=true; ditulis", d / "ROI_APPROVED")

In [ ]:
# ---------------------------------------------------------------- pengendalian kebocoran L1-L6 (dari leakage.py Versi 1)
class LeakageError(AssertionError):
    pass


def check_l1(splits: dict) -> bool:
    # L1: tidak ada pasien di latih/validasi dan uji sekaligus; tiap pasien diuji tepat sekali per ulangan.
    allp = set(splits["all_patients"])
    for r in splits["repeats"]:
        tested = []
        for f in r["folds"]:
            tr, va, te = set(f["train"]), set(f["val"]), set(f["test"])
            if tr & te:
                raise LeakageError(f"L1: pasien latih juga di uji (rep {r['rep']} lipatan {f['fold']}): {sorted(tr & te)[:5]}")
            if va & te:
                raise LeakageError(f"L1: pasien validasi juga di uji (rep {r['rep']} lipatan {f['fold']}): {sorted(va & te)[:5]}")
            if tr & va:
                raise LeakageError(f"L1: pasien latih juga di validasi (rep {r['rep']} lipatan {f['fold']})")
            if tr | va | te != allp:
                raise LeakageError(f"L1: gabungan partisi != seluruh pasien (rep {r['rep']} lipatan {f['fold']})")
            tested += list(te)
        if sorted(tested) != sorted(allp):
            raise LeakageError(f"L1: pada rep {r['rep']} tiap pasien harus diuji tepat sekali")
    return True


def check_l5(splits: dict, resolution_of: dict) -> bool:
    # L5 (S7): semua pasien latih/validasi berresolusi mayoritas, semua pasien uji bukan.
    major = splits["major_resolution"]
    for r in splits["repeats"]:
        for f in r["folds"]:
            tr, va, te = set(f["train"]), set(f["val"]), set(f["test"])
            if (tr | va) & te or tr & va:
                raise LeakageError(f"L5: partisi S7 saling tumpang tindih (rep {r['rep']})")
            bad = [p for p in tr | va if resolution_of[p] != major]
            if bad:
                raise LeakageError(f"L5: pasien non-mayoritas di latih/validasi: {sorted(bad)[:5]}")
            bad = [p for p in te if resolution_of[p] == major]
            if bad:
                raise LeakageError(f"L5: pasien resolusi mayoritas di uji S7: {sorted(bad)[:5]}")
    return True


def check_splits(splits: dict, resolution_of=None) -> bool:
    if splits.get("kind") == "s7":
        if resolution_of is None:
            raise LeakageError("L5: peta resolusi pasien diperlukan untuk memeriksa split S7")
        return check_l5(splits, resolution_of)
    return check_l1(splits)


def check_l6(cfg: dict) -> bool:
    # L6: metrik struktur pembuluh memakai filter berbeda dari pembuat ROI.
    if cfg["gan"]["quality"]["vessel_filter"].lower() == cfg["preprocess"]["roi"]["filter"].lower():
        raise LeakageError("L6: filter metrik pembuluh sama dengan filter pembuat ROI")
    return True


def check_l3_job(job_patients: dict, split_fold: dict, synthetic_source_patients=None) -> bool:
    # L3: pasien yang dipakai job sama dengan partisi split; citra sintetis hanya bersumber dari pasien latih.
    test = set(split_fold["test"])
    used = set(job_patients["train"]) | set(job_patients["val"])
    if used & test:
        raise LeakageError(f"L3: job memakai pasien uji: {sorted(used & test)[:5]}")
    if set(job_patients["train"]) != set(split_fold["train"]) or set(job_patients["val"]) != set(split_fold["val"]):
        raise LeakageError("L3: daftar pasien job berbeda dari berkas split")
    if set(job_patients["test"]) != test:
        raise LeakageError("L3: daftar pasien uji job berbeda dari berkas split")
    if synthetic_source_patients is not None and not set(synthetic_source_patients) <= set(split_fold["train"]):
        raise LeakageError("L3: citra sintetis bersumber dari pasien non-latih")
    return True


def check_l3_dir(jobs_root, scenario: str, splits: dict) -> int:
    n = 0
    for r in splits["repeats"]:
        for f in r["folds"]:
            pj = Path(jobs_root) / scenario / f"rep{r['rep']}_fold{f['fold']}" / "patients.json"
            if pj.exists():
                check_l3_job(json.loads(pj.read_text(encoding="utf-8")), f)
                n += 1
    return n


class GuardedArray:
    # L2: pembungkus larik yang menolak akses ke baris terlarang (indeks uji) dan penyerahan seluruh larik.
    def __init__(self, arr, forbidden):
        self._arr = arr
        self._forbidden = np.asarray(sorted(set(map(int, forbidden))), dtype=np.int64)
        self.accessed = set()

    def __len__(self):
        return len(self._arr)

    @property
    def shape(self):
        return self._arr.shape

    def __array__(self, *a, **k):
        raise LeakageError("L2: seluruh larik diminta pada fase fit; gunakan indeks latih/validasi saja")

    def __getitem__(self, idx):
        if isinstance(idx, slice):
            pos = np.arange(len(self._arr))[idx]
        else:
            idx_a = np.asarray(idx)
            pos = np.flatnonzero(idx_a) if idx_a.dtype == bool else idx_a.ravel().astype(np.int64)
        if np.intersect1d(pos, self._forbidden).size:
            raise LeakageError("L2: fase fit mengakses baris data uji")
        self.accessed.update(int(p) for p in pos)
        return np.asarray(self._arr[idx])

In [ ]:
# ---------------------------------------------------------------- keluaran job, pembacaan split, penjalan job (dari jobio.py, splits.py, jobs.py)
def job_dir(cfg: dict, scenario: str, rep: int, fold: int) -> Path:
    return P(cfg, "jobs", scenario, f"rep{rep}_fold{fold}")


def resolution_map(cfg: dict) -> dict:
    a = pd.read_csv(P(cfg, "cp1", "audit.csv"))
    return dict(zip(a.patient_uid, a.resolution))


def write_run_info(d: Path, cfg: dict, cfg_hash: str, seed: int, extra: dict) -> None:
    atomic_write_json(d / "run_info.json", {
        "scenario": cfg.get("scenario"), "config_hash": cfg_hash, "seed": seed, "git_commit": GIT_COMMIT,
        "torch": torch.__version__, "platform": platform.platform(), **extra})


def write_preds(d: Path, name: str, meta: pd.DataFrame, idx: np.ndarray, probs: np.ndarray, thr: float) -> pd.DataFrame:
    # preds_<name>_images.csv (per citra) dan preds_<name>_patients.csv (rata-rata per pasien, FR-09)
    df = pd.DataFrame({"idx": idx, "patient_uid": meta.patient_uid.to_numpy()[idx],
                       "label": meta.label.to_numpy()[idx], "prob_malignant": probs})
    atomic_write_bytes(d / f"preds_{name}_images.csv", df.to_csv(index=False).encode("utf-8"))
    pat = df.groupby("patient_uid").agg(label=("label", "first"), prob_malignant=("prob_malignant", "mean"),
                                        n_images=("idx", "size")).reset_index()
    pat["pred"] = (pat.prob_malignant >= thr).astype(int)
    atomic_write_bytes(d / f"preds_{name}_patients.csv", pat.to_csv(index=False).encode("utf-8"))
    return pat


def balanced_logloss(y: np.ndarray, p1: np.ndarray, eps: float = 1e-7) -> float:
    p1 = np.clip(p1, eps, 1 - eps)
    losses = [(-np.log(1 - p1[y == 0])).mean() if (y == 0).any() else None,
              (-np.log(p1[y == 1])).mean() if (y == 1).any() else None]
    return float(np.mean([v for v in losses if v is not None]))


# ---------------------------------------------------------------- split: baca
def _split_name(splits_name: str):
    return ("splits.json", "fold_summary.csv") if splits_name == "main" else (f"splits_{splits_name}.json", f"fold_summary_{splits_name}.csv")


def load_splits(cfg: dict, splits_name=None) -> dict:
    fj, _ = _split_name(splits_name or cfg.get("splits_name", "main"))
    return json.loads(P(cfg, "splits", fj).read_text(encoding="utf-8"))


def get_fold(splits: dict, rep: int, fold: int) -> dict:
    return splits["repeats"][rep]["folds"][fold]


# ---------------------------------------------------------------- penjalan job (pengganti jobs.py: pemanggilan fungsi, bukan subproses)
def job_status(cfg: dict, scenario: str) -> list:
    rows = []
    for r in load_splits(cfg)["repeats"]:
        for f in r["folds"]:
            d = job_dir(cfg, scenario, r["rep"], f["fold"])
            rows.append({"rep": r["rep"], "fold": f["fold"], "DONE": (d / "DONE").exists(), "ada_last_pt": (d / "last.pt").exists()})
    return rows


def _selected(splits: dict, reps, folds):
    for r in splits["repeats"]:
        if reps is not None and r["rep"] not in reps:
            continue
        for f in r["folds"]:
            if folds is None or f["fold"] in folds:
                yield r["rep"], f["fold"]


def run_job_safe(label: str, fn, cfg: dict, attempts: int = 2):
    """Jalankan satu job. Hanya RuntimeError (termasuk CUDA OOM) diulang sekali setelah memori GPU dibersihkan; job yang gagal
    dua kali menghentikan antrean dan HARUS dicatat di LOG.md (RULES §4). Tidak ada seed diganti atau lipatan dibuang."""
    log = P(cfg, "jobs", "launch.log")
    log.parent.mkdir(parents=True, exist_ok=True)
    err = None
    for a in range(1, attempts + 1):
        t0 = time.time()
        try:
            res = fn()
            ok = True
        except RuntimeError as e:
            res, ok, err = None, False, repr(e)[:300]
        free_gpu()
        with open(log, "a", encoding="utf-8") as fh:
            fh.write(f"{dt.datetime.now().isoformat(timespec='seconds')}\t{label}\tpercobaan={a}\t{'OK' if ok else 'GAGAL ' + str(err)}\t{time.time() - t0:.0f}s\n")
        if ok:
            return res
    raise RuntimeError(f"Job gagal {attempts}x berturut-turut: {label}. Antrean dihentikan. Catat penyebabnya di LOG.md sebelum dilanjutkan (RULES §4). Terakhir: {err}")


def launch_scenario(config: str, fn, reps=None, folds=None) -> None:
    # satu job per (rep, lipatan): fn(cfg, rep, fold)
    cfg = load_merged(config)
    for r, k in _selected(load_splits(cfg), reps, folds):
        run_job_safe(f"{cfg['scenario']}/rep{r}_fold{k}", lambda r=r, k=k: fn(load_merged(config), r, k), cfg)


def launch_gan_chain(config: str, gan_fn, clf_fn, ratios=None, reps=None, folds=None) -> None:
    # S5/S6/S7: GAN lipatan k, lalu klasifikator tiap rasio (urutan wajib, PRD §8). ratios=[None] untuk S7.
    cfg = load_merged(config)
    sc = cfg["scenario"]
    ratios = ratios if ratios is not None else cfg["gan"]["ratios"]
    for r, k in _selected(load_splits(cfg), reps, folds):
        run_job_safe(f"{sc}_gan/rep{r}_fold{k}", lambda r=r, k=k: gan_fn(load_merged(config), r, k), cfg)
        for ra in ratios:
            def _clf(r=r, k=k, ra=ra):
                c = load_merged(config)
                if ra is not None:
                    c["synthetic"]["ratio"] = ra
                return clf_fn(c, r, k)
            tag = f"_r{int(round(ra * 100))}" if ra is not None else ""
            run_job_safe(f"{sc}{tag}/rep{r}_fold{k}", _clf, cfg)

In [ ]:
# ---------------------------------------------------------------- evaluasi tingkat pasien (dari evaluate.py Versi 1, FR-09, FR-10, FR-13)
# Probabilitas pasien = rata-rata citra; bootstrap persentil bertingkat 2.000 ulangan; selisih berpasangan; perbandingan UTAMA = S6 vs S1/S2.
from sklearn.metrics import roc_auc_score


METRICS = ["auroc", "f1", "recall", "precision", "accuracy"]
PRIMARY = {("S6", "S1"), ("S6", "S2")}


# ===================================================================== metrik dan bootstrap
def patient_metrics(y: np.ndarray, p: np.ndarray, thr: float) -> dict:
    pred = (p >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    return {"auroc": float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else float("nan"),
            "f1": 2 * prec * rec / (prec + rec) if prec + rec else 0.0, "recall": rec, "precision": prec,
            "accuracy": (tp + tn) / len(y), "tp": tp, "fp": fp, "fn": fn, "tn": tn}


def _strata(y: np.ndarray) -> list[np.ndarray]:
    return [np.flatnonzero(y == c) for c in np.unique(y)]


def _boot_indices(strata: list[np.ndarray], rng: np.random.Generator) -> np.ndarray:
    return np.concatenate([rng.choice(s, size=len(s), replace=True) for s in strata])


def bootstrap_ci(y: np.ndarray, p: np.ndarray, thr: float, reps: int, level: float, seed: int) -> dict:
    est = patient_metrics(y, p, thr)
    rng = np.random.default_rng(seed)
    strata = _strata(y)
    draws = {m: [] for m in METRICS}
    for _ in range(reps):
        i = _boot_indices(strata, rng)
        r = patient_metrics(y[i], p[i], thr)
        for m in METRICS:
            draws[m].append(r[m])
    a = (1 - level) / 2 * 100
    out = {}
    for m in METRICS:
        lo, hi = np.nanpercentile(draws[m], [a, 100 - a])
        out[m] = {"estimate": est[m], "ci_low": float(lo), "ci_high": float(hi)}
    out["confusion"] = {k: est[k] for k in ("tp", "fp", "fn", "tn")}
    return out


def paired_bootstrap_diff(y, pa, pb, thr: float, reps: int, level: float, seed: int) -> dict:
    """Selisih A - B pada pasien yang sama. Bermakna bila interval tidak mencakup nol (PRD §3)."""
    ea, eb = patient_metrics(y, pa, thr), patient_metrics(y, pb, thr)
    rng = np.random.default_rng(seed)
    strata = _strata(y)
    d = {m: [] for m in METRICS}
    for _ in range(reps):
        i = _boot_indices(strata, rng)
        ra, rb = patient_metrics(y[i], pa[i], thr), patient_metrics(y[i], pb[i], thr)
        for m in METRICS:
            d[m].append(ra[m] - rb[m])
    a = (1 - level) / 2 * 100
    out = {}
    for m in METRICS:
        lo, hi = np.nanpercentile(d[m], [a, 100 - a])
        out[m] = {"diff": ea[m] - eb[m], "ci_low": float(lo), "ci_high": float(hi), "mencakup_nol": bool(lo <= 0 <= hi)}
    return out


def unpaired_bootstrap_diff(y1, p1, y2, p2, thr: float, reps: int, level: float, seed: int) -> dict:
    """Selisih kelompok 1 - kelompok 2 pada pasien BERBEDA (S7: dalam-distribusi vs pergeseran resolusi)."""
    e1, e2 = patient_metrics(y1, p1, thr), patient_metrics(y2, p2, thr)
    rng = np.random.default_rng(seed)
    s1, s2 = _strata(y1), _strata(y2)
    d = {m: [] for m in METRICS}
    for _ in range(reps):
        i, j = _boot_indices(s1, rng), _boot_indices(s2, rng)
        r1, r2 = patient_metrics(y1[i], p1[i], thr), patient_metrics(y2[j], p2[j], thr)
        for m in METRICS:
            d[m].append(r1[m] - r2[m])
    a = (1 - level) / 2 * 100
    out = {}
    for m in METRICS:
        lo, hi = np.nanpercentile(d[m], [a, 100 - a])
        out[m] = {"diff": e1[m] - e2[m], "ci_low": float(lo), "ci_high": float(hi), "mencakup_nol": bool(lo <= 0 <= hi)}
    return out


# ===================================================================== agregasi job
def collect_predictions(cfg: dict, scenario: str, jobs_root=None, allow_partial: bool = False) -> tuple[pd.DataFrame, dict]:
    """Baca preds_test_patients.csv dari job berstatus DONE. Tanpa allow_partial, semua job wajib selesai."""
    splits = load_splits(cfg)
    rows, expected, found = [], 0, 0
    for r in splits["repeats"]:
        for f in r["folds"]:
            expected += 1
            d = job_dir(cfg, scenario, r["rep"], f["fold"]) if jobs_root is None else Path(jobs_root) / scenario / f"rep{r['rep']}_fold{f['fold']}"
            if (d / "DONE").exists() and (d / "preds_test_patients.csv").exists():
                df = pd.read_csv(d / "preds_test_patients.csv")
                df["rep"], df["fold"] = r["rep"], f["fold"]
                rows.append(df)
                found += 1
    if found < expected and not allow_partial:
        raise RuntimeError(f"{scenario}: hanya {found}/{expected} job DONE; agregasi hanya membaca job selesai (PRD §8)")
    if not rows:
        raise RuntimeError(f"{scenario}: tidak ada job DONE")
    return pd.concat(rows, ignore_index=True), {"jobs_found": found, "jobs_expected": expected}


def aggregate_patients(preds: pd.DataFrame, average_repeats: bool = True) -> pd.DataFrame:
    # average_repeats=True (D4): probabilitas pasien dirata-ratakan atas semua ulangan.
    # False: hanya ulangan pertama yang dipakai (satu prediksi per pasien); n_ulangan tetap dilaporkan = 1.
    if not average_repeats:
        preds = preds[preds["rep"] == preds["rep"].min()]
    g = preds.groupby("patient_uid")
    return g.agg(label=("label", "first"), prob=("prob_malignant", "mean"), n_ulangan=("rep", "nunique")).reset_index()


def fold_table(preds: pd.DataFrame, thr: float) -> pd.DataFrame:
    rows = []
    for (rep, fold), g in preds.groupby(["rep", "fold"]):
        m = patient_metrics(g.label.to_numpy(), g.prob_malignant.to_numpy(), thr)
        rows.append({"rep": rep, "fold": fold, "n_pasien": len(g), "n_ganas": int(g.label.sum()), **m})
    return pd.DataFrame(rows)


def evaluate_preds(cfg: dict, scenario: str, preds: pd.DataFrame, cov: dict, extra_note: str = "") -> dict:
    ev = cfg["eval"]
    pat = aggregate_patients(preds, ev["average_repeats"])
    y, p = pat.label.to_numpy(), pat.prob.to_numpy()
    res = bootstrap_ci(y, p, ev["threshold"], ev["bootstrap_reps"], ev["ci_level"], ev["bootstrap_seed"])
    note = "PARSIAL: tidak semua job selesai" if cov["jobs_found"] < cov["jobs_expected"] else \
        "lengkap; interval = bootstrap persentil bertingkat atas pasien"
    rows = [{"scenario": scenario, "metric": m, "estimate": res[m]["estimate"], "ci_low": res[m]["ci_low"],
             "ci_high": res[m]["ci_high"], "n_pasien": len(pat), "n_ganas": int(y.sum()),
             "job_selesai": f"{cov['jobs_found']}/{cov['jobs_expected']}", "catatan": (note + " " + extra_note).strip()}
            for m in METRICS]
    out_dir = P(cfg, "results")
    atomic_write_bytes(out_dir / f"results_{scenario}.csv", pd.DataFrame(rows).to_csv(index=False).encode("utf-8"))
    atomic_write_bytes(out_dir / f"fold_metrics_{scenario}.csv", fold_table(preds, ev["threshold"]).to_csv(index=False).encode("utf-8"))
    atomic_write_bytes(out_dir / f"patient_probs_{scenario}.csv", pat.to_csv(index=False).encode("utf-8"))
    return {"scenario": scenario, "coverage": cov, "table": pd.DataFrame(rows), "confusion": res["confusion"], "patients": pat}


def evaluate_scenario(cfg: dict, scenario: str, allow_partial: bool = False) -> dict:
    preds, cov = collect_predictions(cfg, scenario, allow_partial=allow_partial)
    return evaluate_preds(cfg, scenario, preds, cov)


def load_result(cfg: dict, scenario: str) -> dict:
    return {"scenario": scenario, "patients": pd.read_csv(P(cfg, "results", f"patient_probs_{scenario}.csv"))}


def compare_scenarios(cfg: dict, a: dict, b: dict, label: str | None = None) -> pd.DataFrame:
    """Selisih berpasangan A - B. Label otomatis: 'utama' hanya untuk S6 vs S1/S2; selain itu 'eksploratif'."""
    ev = cfg["eval"]
    label = label or ("utama" if (a["scenario"], b["scenario"]) in PRIMARY else "eksploratif")
    m = a["patients"].merge(b["patients"], on="patient_uid", suffixes=("_a", "_b"))
    assert (m.label_a == m.label_b).all(), "label pasien tidak konsisten antar skenario"
    d = paired_bootstrap_diff(m.label_a.to_numpy(), m.prob_a.to_numpy(), m.prob_b.to_numpy(), ev["threshold"],
                              ev["bootstrap_reps"], ev["ci_level"], ev["bootstrap_seed"])
    rows = [{"A": a["scenario"], "B": b["scenario"], "metric": k, "selisih_A_minus_B": v["diff"], "ci_low": v["ci_low"],
             "ci_high": v["ci_high"], "mencakup_nol": v["mencakup_nol"], "n_pasien": len(m), "status": label,
             "utama_untuk_metrik": k in ("auroc", "f1") if label == "utama" else False} for k, v in d.items()]
    df = pd.DataFrame(rows)
    atomic_write_bytes(P(cfg, "results", f"compare_{a['scenario']}_vs_{b['scenario']}.csv"), df.to_csv(index=False).encode("utf-8"))
    return df


# ===================================================================== rasio sintetis (G2, L4)
def _min_val_loss(cfg: dict, scenario: str, rep: int, fold: int) -> float | None:
    h = job_dir(cfg, scenario, rep, fold) / "history.csv"
    if not (job_dir(cfg, scenario, rep, fold) / "DONE").exists() or not h.exists():
        return None
    return float(pd.read_csv(h).val_loss_balanced.min())


def assemble_selected(cfg: dict, base: str, ratios: list[float], allow_partial: bool = False) -> tuple[pd.DataFrame, dict]:
    """Untuk tiap (rep, lipatan) pilih rasio dengan val_loss_balanced terendah (pasien VALIDASI), ambil prediksi UJI-nya."""
    splits = load_splits(cfg)
    rows, sel, expected, found = [], [], 0, 0
    for r in splits["repeats"]:
        for f in r["folds"]:
            expected += 1
            vals = {ra: _min_val_loss(cfg, f"{base}_r{int(round(ra * 100))}", r["rep"], f["fold"]) for ra in ratios}
            ok = {k: v for k, v in vals.items() if v is not None}
            if len(ok) < len(ratios) and not allow_partial:
                raise RuntimeError(f"{base} rep{r['rep']} fold{f['fold']}: hanya {len(ok)}/{len(ratios)} rasio selesai")
            if not ok:
                continue
            best = min(ok, key=lambda k: (ok[k], k))
            d = job_dir(cfg, f"{base}_r{int(round(best * 100))}", r["rep"], f["fold"])
            df = pd.read_csv(d / "preds_test_patients.csv")
            df["rep"], df["fold"], df["ratio"] = r["rep"], f["fold"], best
            rows.append(df)
            sel.append({"rep": r["rep"], "fold": f["fold"], "ratio_terpilih": best,
                        **{f"val_loss_r{int(round(k * 100))}": v for k, v in ok.items()},
                        "sumber_pemilihan": "val_loss_balanced pasien validasi dalam lipatan latih"})
            found += 1
    if not rows:
        raise RuntimeError(f"{base}: tidak ada job rasio yang DONE")
    atomic_write_bytes(P(cfg, "results", f"ratio_selection_{base}.csv"), pd.DataFrame(sel).to_csv(index=False).encode("utf-8"))
    return pd.concat(rows, ignore_index=True), {"jobs_found": found, "jobs_expected": expected}


def evaluate_selected(cfg: dict, base: str, allow_partial: bool = False) -> dict:
    ratios = cfg["gan"]["ratios"]
    preds, cov = assemble_selected(cfg, base, ratios, allow_partial)
    return evaluate_preds(cfg, base, preds, cov, extra_note="rasio dipilih per lipatan dari validasi (G2, L4)")


def evaluate_all_ratios(cfg: dict, base: str, allow_partial: bool = False) -> pd.DataFrame:
    """Semua rasio dilaporkan (G2: r=25%, 50%, 100% semuanya)."""
    out = []
    for ra in cfg["gan"]["ratios"]:
        r = evaluate_scenario(cfg, f"{base}_r{int(round(ra * 100))}", allow_partial)
        out.append(r["table"])
    return pd.concat(out, ignore_index=True)


def select_best_config(cfg: dict, bases: tuple[str, ...] = ("S5", "S6")) -> dict:
    """INFORMASI untuk CV utama (rasio dengan rata-rata val_loss_balanced terendah). S7 TIDAK memakainya: validasi CV utama
    melibatkan pasien resolusi minor yang menjadi pasien uji S7 (L5). Lihat select_s7_ratio."""
    splits = load_splits(cfg, "main")
    out = {}
    for base in bases:
        means = {}
        for ra in cfg["gan"]["ratios"]:
            v = [_min_val_loss(cfg, f"{base}_r{int(round(ra * 100))}", r["rep"], f["fold"]) for r in splits["repeats"] for f in r["folds"]]
            if any(x is None for x in v):
                raise RuntimeError(f"{base}_r{int(round(ra * 100))}: ada job belum DONE; konfigurasi terbaik butuh semua lipatan")
            means[ra] = float(np.mean(v))
        best = min(means, key=lambda k: (means[k], k))
        out[base] = {"ratio": best, "mean_val_loss_balanced": {str(k): v for k, v in means.items()},
                     "source": "val_loss_balanced pasien validasi dalam lipatan latih (L4); uji tidak dipakai"}
    sel = P(cfg, "cp4", "lambda_selection.json")
    if sel.exists() and "S6" in out:
        out["S6"]["lambda_roi"] = json.loads(sel.read_text(encoding="utf-8"))["chosen"]
    atomic_write_json(P(cfg, "results", "best_config.json"), out)
    return out


# ===================================================================== S7
def select_s7_ratio(cfg: dict, base: str, allow_partial: bool = False) -> dict:
    """L4/L5: rasio S7 dipilih HANYA dari val_loss_balanced pasien validasi di dalam split S7 (resolusi mayoritas),
    dirata-ratakan atas ulangan S7. Satu rasio untuk semua ulangan. Pasien uji S7 (resolusi lain) tidak ikut."""
    splits = load_splits(cfg, "s7")
    rows, means = [], {}
    for ra in cfg["gan"]["ratios"]:
        v = [(r["rep"], _min_val_loss(cfg, f"{base}_r{int(round(ra * 100))}", r["rep"], f["fold"])) for r in splits["repeats"] for f in r["folds"]]
        ok = [x for _, x in v if x is not None]
        if len(ok) < len(v) and not allow_partial:
            raise RuntimeError(f"{base}_r{int(round(ra * 100))}: hanya {len(ok)}/{len(v)} job DONE; pemilihan rasio S7 butuh semuanya")
        if ok:
            means[ra] = float(np.mean(ok))
        rows += [{"ratio": ra, "rep": rp, "val_loss_balanced": x} for rp, x in v]
    if not means:
        raise RuntimeError(f"{base}: tidak ada job rasio S7 yang DONE")
    best = min(means, key=lambda k: (means[k], k))
    out = {"base": base, "ratio": best, "mean_val_loss_balanced": {str(k): v for k, v in means.items()},
           "source": "val_loss_balanced pasien validasi DI DALAM split S7 (resolusi mayoritas); pasien uji S7 dan CV utama tidak dipakai (L5)"}
    atomic_write_bytes(P(cfg, "results", f"ratio_selection_{base}.csv"), pd.DataFrame(rows).to_csv(index=False).encode("utf-8"))
    atomic_write_json(P(cfg, "results", f"best_ratio_{base}.json"), out)
    return out


def evaluate_s7_selected(cfg: dict, base: str, allow_partial: bool = False) -> dict:
    sel = select_s7_ratio(cfg, base, allow_partial)
    ra = sel["ratio"]
    preds, cov = collect_predictions(cfg, f"{base}_r{int(round(ra * 100))}", allow_partial=allow_partial)
    return evaluate_preds(cfg, base, preds, cov, extra_note=f"rasio {ra} dipilih dari validasi dalam split S7 (L4, L5)")


def s7_gap(cfg: dict, main_scenario: str, s7_scenario: str) -> pd.DataFrame:
    """Selisih kinerja dalam-distribusi (pasien resolusi mayoritas pada CV utama) vs pergeseran (uji gabungan S7).
    Pasien berbeda dan kondisi pelatihan tidak persis sama (CV utama juga melatih pada resolusi lain) -> EKSPLORATIF."""
    ev = cfg["eval"]
    audit = pd.read_csv(P(cfg, "cp1", "audit.csv"))
    major = load_splits(cfg, "s7")["major_resolution"]
    a = pd.read_csv(P(cfg, "results", f"patient_probs_{main_scenario}.csv")).merge(audit[["patient_uid", "resolution"]], on="patient_uid")
    ind = a[a.resolution == major]
    shifted = pd.read_csv(P(cfg, "results", f"patient_probs_{s7_scenario}.csv"))
    d = unpaired_bootstrap_diff(ind.label.to_numpy(), ind.prob.to_numpy(), shifted.label.to_numpy(), shifted.prob.to_numpy(),
                                ev["threshold"], ev["bootstrap_reps"], ev["ci_level"], ev["bootstrap_seed"])
    e_in = patient_metrics(ind.label.to_numpy(), ind.prob.to_numpy(), ev["threshold"])
    e_sh = patient_metrics(shifted.label.to_numpy(), shifted.prob.to_numpy(), ev["threshold"])
    rows = [{"metode": main_scenario, "metric": m, "dalam_distribusi": e_in[m], "pergeseran_S7": e_sh[m],
             "selisih": v["diff"], "ci_low": v["ci_low"], "ci_high": v["ci_high"], "mencakup_nol": v["mencakup_nol"],
             "n_dalam": len(ind), "n_pergeseran": len(shifted), "status": "eksploratif"} for m, v in d.items()]
    df = pd.DataFrame(rows)
    atomic_write_bytes(P(cfg, "results", f"s7_gap_{main_scenario}.csv"), df.to_csv(index=False).encode("utf-8"))
    return df


# ===================================================================== tabel (FR-13)
ORDER = ["S1", "S2", "S3_svm", "S3_lda", "S4_svm", "S4_lda", "S5", "S6", "S7_S1", "S7_S5", "S7_S6",
         "S3_svm_a01", "S3_lda_a01", "S4_svm_a01", "S4_lda_a01"]


def make_tables(cfg: dict) -> dict:
    res = []
    for sc in ORDER + [f"{b}_r{int(round(r * 100))}" for b in ("S5", "S6") for r in cfg["gan"]["ratios"]]:
        p = P(cfg, "results", f"results_{sc}.csv")
        if p.exists():
            res.append(pd.read_csv(p))
    if not res:
        raise RuntimeError("belum ada results_*.csv")
    allr = pd.concat(res, ignore_index=True)
    allr["sel"] = allr.apply(lambda r: f"{r.estimate:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}]", axis=1)
    wide = allr.pivot(index="scenario", columns="metric", values="sel")[METRICS]
    meta = allr.groupby("scenario").agg(n_pasien=("n_pasien", "first"), job_selesai=("job_selesai", "first"),
                                        catatan=("catatan", "first"))
    t33 = wide.join(meta)
    t33["status"] = ["eksploratif (sensitivitas LASSO)" if s.endswith("_a01") else
                     ("rincian rasio" if "_r" in s else "") for s in t33.index]
    t33 = t33.reindex([s for s in ORDER if s in t33.index] + [s for s in t33.index if s not in ORDER])
    atomic_write_bytes(P(cfg, "results", "tabel_3_3.csv"), t33.reset_index().to_csv(index=False).encode("utf-8"))
    atomic_write_bytes(P(cfg, "results", "tabel_3_3.md"), _df_to_md(t33.reset_index()).encode("utf-8"))

    lit = yaml.safe_load(open(ROOT / "configs" / "literature.yaml", encoding="utf-8"))["rows"]
    t34 = pd.DataFrame(lit)
    ours = []
    for sc in ("S1", "S2", "S6"):
        r = allr[allr.scenario == sc]
        if not r.empty:
            f1 = r[r.metric == "f1"].iloc[0]
            ours.append({"sumber": f"Penelitian ini ({sc})", "data": "CE-NBI 210 pasien, CV 5x2 per pasien",
                         "metrik": "F1 pasien [CI 95%]", "nilai_validasi_silang": f"{f1.estimate:.3f} [{f1.ci_low:.3f}, {f1.ci_high:.3f}]",
                         "nilai_uji": None, "catatan": "interval = bootstrap pasien; bukan median fold seperti Wang dkk."})
    t34 = pd.concat([t34, pd.DataFrame(ours)], ignore_index=True)
    atomic_write_bytes(P(cfg, "results", "tabel_3_4.csv"), t34.to_csv(index=False).encode("utf-8"))
    return {"tabel_3_3": t33, "tabel_3_4": t34}


def _df_to_md(df: pd.DataFrame) -> str:
    # Tabel Markdown sungguhan, dengan atau tanpa tabulate (versi lama menulis CSV ke berkas .md bila tabulate tidak ada).
    cells = [[str(c) for c in df.columns]] + [[("" if pd.isna(v) else str(v)) for v in row] for row in df.itertuples(index=False)]
    head, rows = cells[0], cells[1:]
    return "\n".join(["| " + " | ".join(head) + " |", "| " + " | ".join("---" for _ in head) + " |"] + ["| " + " | ".join(r) + " |" for r in rows]) + "\n"


def _has_tabulate() -> bool:
    try:
        import tabulate  # noqa: F401
        return True
    except ImportError:
        return False


def run_comparisons(cfg: dict) -> pd.DataFrame:
    """Perbandingan utama (S6 vs S1, S6 vs S2) + eksploratif lain yang hasilnya tersedia."""
    pairs = [("S6", "S1"), ("S6", "S2"), ("S6", "S5"), ("S5", "S1"), ("S1", "S2"), ("S6", "S3_svm"), ("S6", "S3_lda"),
             ("S6", "S4_svm"), ("S6", "S4_lda")]
    out = []
    for a, b in pairs:
        if (P(cfg, "results", f"patient_probs_{a}.csv").exists() and P(cfg, "results", f"patient_probs_{b}.csv").exists()):
            out.append(compare_scenarios(cfg, load_result(cfg, a), load_result(cfg, b)))
    if not out:
        raise RuntimeError("belum ada pasangan skenario yang hasilnya lengkap")
    df = pd.concat(out, ignore_index=True)
    atomic_write_bytes(P(cfg, "results", "comparisons_all.csv"), df.to_csv(index=False).encode("utf-8"))
    return df

In [ ]:
# ---------------------------------------------------------------- pemeriksaan checkpoint (dari checkpoint.py Versi 1)
# Prinsip: item yang tidak bisa dijalankan dilaporkan apa adanya (GAGAL / TIDAK_TERSEDIA / MENUNGGU), tidak pernah diubah menjadi LULUS.
# `lulus` benar hanya bila tidak ada GAGAL atau MENUNGGU. manifest.json ditulis hanya bila lulus. Tag git TIDAK dibuat di sini.
def _scenario_items(cfg: dict, sc: str, label=None):
    # Item standar satu skenario: semua job DONE, L3 lolos, tabel hasil lengkap dengan interval.
    label = label or sc
    items, files = [], []
    st = job_status(cfg, sc)
    n_done = sum(r["DONE"] for r in st)
    items.append(_item(f"{label}: seluruh lipatan berstatus DONE", LULUS if n_done == len(st) else GAGAL, f"{n_done}/{len(st)}"))
    try:
        n = check_l3_dir(P(cfg, "jobs"), sc, load_splits(cfg))
        items.append(_item(f"{label}: L3 daftar pasien tiap job", LULUS, f"{n} job"))
    except LeakageError as e:
        items.append(_item(f"{label}: L3", GAGAL, str(e)))
    rp = P(cfg, "results", f"results_{sc}.csv")
    if rp.exists():
        r = pd.read_csv(rp)
        full = r.job_selesai.str.split("/").map(lambda a: a[0] == a[1]).all()
        items.append(_item(f"{label}: tabel hasil dengan interval kepercayaan", LULUS if full else GAGAL,
                           "; ".join(f"{x.metric}={x.estimate:.3f} [{x.ci_low:.3f}, {x.ci_high:.3f}]" for x in r.itertuples())
                           + f" | job {r.job_selesai.iloc[0]}"))
        files += [rp, P(cfg, "results", f"fold_metrics_{sc}.csv"), P(cfg, "results", f"patient_probs_{sc}.csv")]
    else:
        items.append(_item(f"{label}: tabel hasil", GAGAL, f"{rp.name} tidak ada"))
    return items, files


def finalize_cp7(cfg: dict) -> dict:
    out = P(cfg, "cp7")
    items, files = [], []
    res = P(cfg, "results")
    for n in ("tabel_3_3.csv", "tabel_3_4.csv", "comparisons_all.csv"):
        items.append(_item(f"{n} dibuat otomatis dari hasil job", LULUS if (res / n).exists() else GAGAL, n))
        files.append(res / n)
    cp = res / "comparisons_all.csv"
    if cp.exists():
        c = pd.read_csv(cp)
        main_ = c[c.status == "utama"]
        need = {("S6", "S1"), ("S6", "S2")}
        have = set(zip(main_.A, main_.B))
        items.append(_item("Perbandingan utama S6 vs S1 dan S2 dilaporkan apa pun arahnya (AUROC, F1)", LULUS if need <= have else GAGAL,
                           "; ".join(f"{r.A} vs {r.B} {r.metric}: {r.selisih_A_minus_B:+.3f} [{r.ci_low:+.3f}, {r.ci_high:+.3f}]"
                                     f"{' (mencakup nol)' if r.mencakup_nol else ' (tidak mencakup nol)'}"
                                     for r in main_[main_.metric.isin(['auroc', 'f1'])].itertuples())))
        others = c[c.status != "utama"]
        items.append(_item("Perbandingan lain berlabel eksploratif", LULUS if (others.status == "eksploratif").all() else GAGAL,
                           f"{others[['A', 'B']].drop_duplicates().shape[0]} pasangan"))
    bad = []
    for f in sorted(res.glob("results_*.csv")):
        r = pd.read_csv(f)
        if not r.job_selesai.str.split("/").map(lambda a: a[0] == a[1]).all():
            bad.append(f.name)
    items.append(_item("Setiap angka terlacak ke job selesai (tidak ada tabel PARSIAL)", LULUS if not bad else GAGAL,
                       "semua results_*.csv lengkap" if not bad else f"parsial: {bad}"))
    items.append(_item("Semua tabel melaporkan interval kepercayaan", LULUS, "kolom ci_low/ci_high pada results_*.csv"))
    items.append(_review_item(out))
    return _finish("CP7", out, items, files)

## 3. Perbandingan berpasangan (bootstrap pasien yang sama)

In [ ]:
c = run_comparisons(cfg)
display(c[c.status == 'utama'])
print('\nEksploratif:')
display(c[c.status != 'utama'])

## 4. Tabel 3.3 dan Tabel 3.4 (dibuat otomatis dari hasil job)

Tabel 3.4 memuat baris literatur dari `configs/literature.yaml`; lengkapi dari proposal. Tidak ada angka yang dikarang.

In [ ]:
t = make_tables(cfg)
display(t['tabel_3_3'])
display(t['tabel_3_4'])

## 5. Pemeriksaan checkpoint CP7 dan `manifest.json` (sel terakhir)

In [ ]:
# Isi nama peninjau KEDUA setelah benar-benar meninjau hasil di atas (RULES §3), lalu jalankan sel ini.
REVIEWER = None      # contoh: 'Ahmad'
if REVIEWER:
    approve_review('CP7', REVIEWER)

In [ ]:
report = finalize_cp7(cfg)
assert report['checkpoint'] == 'CP7'